# Stage 19 Transition-Regularization Ablation

This notebook launches the validation-only Stage 19 ablation for the Stage 16 61-epoch frozen-embedding TCN ensemble. It estimates the transition penalty from training labels only, reuses the completed Stage 16 ensemble as `lambda_transition = 0.0`, and trains the nonzero lambda grid only when the guarded run flag is enabled. The held-out test split is not evaluated here.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.transition_regularization import (
    DEFAULT_STAGE16_REPLICATION_OUTPUT_DIR,
    DEFAULT_STAGE19_OUTPUT_DIR,
    STAGE19_LAMBDAS,
    STAGE19_SEEDS,
    run_stage19_transition_regularization,
)

stage16_replication_dir = repo_root / DEFAULT_STAGE16_REPLICATION_OUTPUT_DIR
stage19_output_dir = repo_root / DEFAULT_STAGE19_OUTPUT_DIR

{
    "repo_root": str(repo_root),
    "stage16_replication_dir_exists": stage16_replication_dir.exists(),
    "stage19_output_dir": str(stage19_output_dir),
    "lambdas": list(STAGE19_LAMBDAS),
    "seeds": list(STAGE19_SEEDS),
}


## Run Guard

Set `RUN_STAGE19_TRANSITION_REGULARIZATION = True` to train the nine nonzero-lambda seed runs and build the per-lambda ensembles. Completed seed runs are reused when their summary and validation prediction artifacts already exist.

In [ ]:
RUN_STAGE19_TRANSITION_REGULARIZATION = False

stage19_summary = pd.DataFrame()
if RUN_STAGE19_TRANSITION_REGULARIZATION:
    stage19_summary = run_stage19_transition_regularization(
        output_dir=stage19_output_dir,
        stage16_replication_dir=stage16_replication_dir,
        lambdas=STAGE19_LAMBDAS,
        seeds=STAGE19_SEEDS,
        skip_completed=True,
    )
    display(stage19_summary)
    print("outputs:", stage19_output_dir)
else:
    summary_path = stage19_output_dir / "experiment_summary.csv"
    if summary_path.exists():
        stage19_summary = pd.read_csv(summary_path)
        display(stage19_summary)
    else:
        print("Stage 19 transition-regularization run is configured but not run.")


## Baseline Comparison

After the guarded run completes, this cell displays the lambda-zero baseline and nonzero-lambda ensemble deltas saved by the Stage 19 helper.

In [ ]:
comparison_path = stage19_output_dir / "baseline_comparison.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    display(
        comparison[
            [
                "lambda_transition",
                "stage19_run_status",
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "Wake_f1",
                "Non_REM_f1",
                "REM_f1",
                "macro_f1_delta_vs_lambda_0",
                "predicted_wake_to_rem_transition_count",
                "true_wake_to_rem_transition_count",
            ]
        ]
    )
else:
    print("No Stage 19 baseline comparison has been written yet.")
